In [1]:
import pandas as pd

In [2]:
data=pd.read_csv("preprocessed_data.csv")

In [3]:
data

,Unnamed: 0,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_days,bmi
0,0,1,0.451905,-0.850650,-0.992245,-0.141118,0,0,0,0,1,0,50,-1.051881
1,1,0,-1.058874,0.767576,0.799848,0.904171,2,0,0,0,1,1,55,1.434887
2,2,0,0.074210,-0.709934,0.202484,-1.186407,2,0,0,0,0,1,51,-0.756267
3,3,1,0.577803,0.556503,1.397213,1.949459,0,0,0,0,1,1,48,0.241981
4,4,0,-1.058874,-1.272795,-1.589609,-2.231695,0,0,0,0,0,0,47,-0.851556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68690,69995,1,0.451905,0.134357,-0.394881,-0.141118,0,0,1,0,1,0,52,-0.100135
68691,69996,0,-0.807077,3.652238,0.799848,0.904171,1,1,0,0,1,1,61,4.417538
68692,69997,1,2.340378,2.174728,3.189306,0.904171,2,0,0,1,0,1,52,0.749118
68693,69998,0,-0.177586,-0.147073,0.501166,-0.141118,0,1,0,0,0,1,61,-0.067169


In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

X = data[[
    "gender", "height", "weight",
    "ap_hi", "ap_lo",
    "cholesterol", "gluc",
    "smoke", "alco", "active",
    "age_days", "bmi"
]].values

y = data["cardio"].values

# ensure binary labels (0/1)
y = (y == 1).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --------------------------------------------------
# 2. STANDARDIZATION
# --------------------------------------------------
def standardize(X, mean=None, std=None):
    if mean is None:
        mean = X.mean(axis=0)
    if std is None:
        std = X.std(axis=0) + 1e-9
    return (X - mean) / std, mean, std

X_train, mean, std = standardize(X_train)
X_test, _, _ = standardize(X_test, mean, std)

# --------------------------------------------------
# 3. LOGISTIC REGRESSION
# --------------------------------------------------
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def logistic_regression_train(X, y, lr=0.05, epochs=3000):
    m, n = X.shape
    weights = np.zeros(n)
    bias = 0

    for _ in range(epochs):
        z = np.dot(X, weights) + bias
        y_hat = sigmoid(z)

        dw = (1/m) * np.dot(X.T, (y_hat - y))
        db = (1/m) * np.sum(y_hat - y)

        weights -= lr * dw
        bias -= lr * db

    return weights, bias

def logistic_regression_predict(X, weights, bias):
    z = np.dot(X, weights) + bias
    probs = sigmoid(z)
    return probs

# --------------------------------------------------
# 4. METRICS
# --------------------------------------------------
def confusion_matrix_manual(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp, tn, fp, fn

def accuracy_score_manual(y_true, y_pred):
    return np.mean(y_true == y_pred)

def precision_score_manual(y_true, y_pred):
    tp, _, fp, _ = confusion_matrix_manual(y_true, y_pred)
    return tp / (tp + fp + 1e-9)

def recall_score_manual(y_true, y_pred):
    tp, _, _, fn = confusion_matrix_manual(y_true, y_pred)
    return tp / (tp + fn + 1e-9)

def f1_score_manual(y_true, y_pred):
    p = precision_score_manual(y_true, y_pred)
    r = recall_score_manual(y_true, y_pred)
    return 2 * p * r / (p + r + 1e-9)

# --------------------------------------------------
# 5. BEST THRESHOLD SEARCH
# --------------------------------------------------
def find_best_threshold(y_true, y_prob):
    best_acc = -1
    best_thresh = 0.5

    for t in np.linspace(0.1, 0.9, 81):
        y_pred = (y_prob >= t).astype(int)
        acc = accuracy_score_manual(y_true, y_pred)

        if acc > best_acc:
            best_acc = acc
            best_thresh = t

    return best_thresh, best_acc

# --------------------------------------------------
# 6. TRAIN MODEL
# --------------------------------------------------
weights, bias = logistic_regression_train(
    X_train,
    y_train,
    lr=0.01,      # smaller LR
    epochs=5000   # more epochs
)

# --------------------------------------------------
# 7. PREDICT PROBABILITIES
# --------------------------------------------------
y_prob = logistic_regression_predict(X_test, weights, bias)

print("Probability range:", y_prob.min(), "to", y_prob.max())

# --------------------------------------------------
# 8. FIND BEST THRESHOLD
# --------------------------------------------------
best_thresh, best_acc = find_best_threshold(y_test, y_prob)

# --------------------------------------------------
# 9. FINAL PREDICTION
# --------------------------------------------------
y_pred = (y_prob >= best_thresh).astype(int)

# --------------------------------------------------
# 10. FINAL METRICS
# --------------------------------------------------
acc = accuracy_score_manual(y_test, y_pred)
precision = precision_score_manual(y_test, y_pred)
recall = recall_score_manual(y_test, y_pred)
f1 = f1_score_manual(y_test, y_pred)

print("\nBest Threshold:", best_thresh)
print("Accuracy :", acc)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)


Probability range: 0.008931732915512641 to 0.9996099279361021

Best Threshold: 0.48
Accuracy : 0.7249435912366257
Precision: 0.7370399373530561
Recall   : 0.6914487217160312
F1 Score : 0.7135167912523531
